In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [8]:
import os
import pandas as pd
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv(PROJECT_ROOT / ".env")

client_id = os.getenv("CLIENT_ID")

In [7]:
anime_data_client = AnimeDataClient(
    client_id,
    cache_file=PROJECT_ROOT / "anime_cache.json",
)

In [10]:
anime_data = anime_data_client.get_cache()

Build features

In [11]:
from anime_features import AnimeFeatureBuilder

builder = AnimeFeatureBuilder(
    anime_data,
    max_tfidf_features=3000,
    n_svd_components=300
)

anime_df = builder.build_features()

builder.svd_explained_variance

np.float64(0.4104185921487374)

Convert each anime in df to vectors

In [12]:
recommender = SimilarityRecommender()
anime_vectors = recommender.create_anime_vectors(anime_df)
anime_df_scaled = recommender.anime_df_scaled

Get user Data

In [13]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

Bayesion Ridge Regression

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.linear_model import BayesianRidge
import numpy as np

rated_items, anime_df, anime_vectors, anime_df_scaled, builder = anime_data_client.get_rated_items(
    user_scores=user_scores,
    anime_data=anime_data,
    builder=builder,
    recommender=recommender,
    anime_df=anime_df,
    anime_vectors=anime_vectors,
)

X = np.array([vec for _, vec, _ in rated_items])
y = np.array([score for _, _, score in rated_items], dtype=float)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=None
)

model = BayesianRidge()
model.fit(X_train, y_train)

pred, pred_std = model.predict(X_test, return_std=True)
pred = np.clip(pred, 1, 10)

mae = mean_absolute_error(y_test, pred)
rmse = root_mean_squared_error(y_test, pred)

baseline_pred = np.full_like(y_test, y_train.mean(), dtype=float)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"Baseline MAE: {baseline_mae:.3f}")
print(f"Improvement vs baseline: {baseline_mae - mae:.3f}")

MAE: 1.413
RMSE: 1.739
Baseline MAE: 1.485
Improvement vs baseline: 0.072


In [16]:
print("num ratings:", len(y))
print("mean:", y.mean())
print("median:", np.median(y))
print("std:", y.std())
print(pd.Series(y).value_counts().sort_index())

num ratings: 253
mean: 7.430830039525691
median: 7.0
std: 1.5502639399039124
1.0      1
3.0      2
4.0      7
5.0      8
6.0     56
7.0     56
8.0     47
9.0     59
10.0    17
Name: count, dtype: int64


Hit Rate

In [17]:
# Hide some high-rated anime, train on the rest, and see if the hidden likes show up near the top.
hit_rating_threshold = np.median(y) + 0.5 * np.std(y)
heldout_fraction = 0.25
top_ks = [5, 10, 20, 50, 100]

rated_eval = pd.DataFrame({
    "anime_id": [anime_id for anime_id, _, _ in rated_items],
    "score": [score for _, _, score in rated_items],
})

liked_eval = rated_eval[rated_eval["score"] >= hit_rating_threshold]
if len(liked_eval) < 2:
    raise ValueError("Need at least 2 high-rated anime to run a hit-rate holdout test.")

heldout_liked = liked_eval.sample(frac=heldout_fraction, random_state=None)
heldout_ids = set(heldout_liked["anime_id"])

train_eval = rated_eval[~rated_eval["anime_id"].isin(heldout_ids)]
train_ids = train_eval["anime_id"].tolist()

X_hit_train = anime_df_scaled.loc[train_ids].to_numpy()
y_hit_train = train_eval["score"].to_numpy(dtype=float)

hit_model = BayesianRidge()
hit_model.fit(X_hit_train, y_hit_train)

# Candidates are everything the model did not train on, including the hidden liked anime.
candidate_ids = [anime_id for anime_id in anime_df_scaled.index if anime_id not in set(train_ids)]
X_hit_candidates = anime_df_scaled.loc[candidate_ids].to_numpy()

hit_pred, hit_pred_std = hit_model.predict(X_hit_candidates, return_std=True)
hit_pred = np.clip(hit_pred, 1, 10)

title_by_id = {int(anime["id"]): anime["title"] for anime in anime_data.values()}
hit_recommendations = pd.DataFrame({
    "anime_id": candidate_ids,
    "title": [title_by_id.get(int(anime_id), "Unknown") for anime_id in candidate_ids],
    "actual_score": [user_scores.get(int(anime_id)) for anime_id in candidate_ids],
    "predicted_score": hit_pred,
    "uncertainty_raw": hit_pred_std,
})

uncertainty_clipped = hit_recommendations["uncertainty_raw"].clip(
    lower=hit_recommendations["uncertainty_raw"].quantile(0.05),
    upper=hit_recommendations["uncertainty_raw"].quantile(0.95),
)
if uncertainty_clipped.max() == uncertainty_clipped.min():
    hit_recommendations["uncertainty"] = 0.0
else:
    hit_recommendations["uncertainty"] = (
        (uncertainty_clipped - uncertainty_clipped.min())
        / (uncertainty_clipped.max() - uncertainty_clipped.min())
    )

hit_recommendations["ranking_score"] = (
    hit_recommendations["predicted_score"] - 4.5 * hit_recommendations["uncertainty"]
)
hit_recommendations["is_hidden_like"] = hit_recommendations["anime_id"].isin(heldout_ids)
hit_recommendations = hit_recommendations.sort_values("ranking_score", ascending=False)

hit_rows = []
for k in top_ks:
    top_k = hit_recommendations.head(k)
    hits = int(top_k["is_hidden_like"].sum())
    hit_rows.append({
        "k": k,
        "hits": hits,
        "heldout_likes": len(heldout_ids),
        "hit_rate": hits / len(heldout_ids),
        "precision_at_k": hits / k,
    })

hit_rate_results = pd.DataFrame(hit_rows)

print(f"Total rated anime: {len(rated_eval)}")
print(f"High-rated anime (score >= {hit_rating_threshold}): {len(liked_eval)}")
print(f"Hidden liked anime: {len(heldout_ids)}")
display(hit_rate_results)

# hit_recommendations[hit_recommendations["is_hidden_like"]].head(20)

Total rated anime: 253
High-rated anime (score >= 7.775131969951956): 123
Hidden liked anime: 31


,k,hits,heldout_likes,hit_rate,precision_at_k
0,5,2,31,0.064516,0.40
1,10,2,31,0.064516,0.20
2,20,5,31,0.161290,0.25
3,50,6,31,0.193548,0.12
4,100,9,31,0.290323,0.09


In [19]:
baseline_recommendations = hit_recommendations.copy()

# If "mean" is available in anime_df, rank by global MAL mean.
baseline_recommendations["baseline_score"] = anime_df.loc[
    baseline_recommendations["anime_id"], "mean"
].to_numpy()

baseline_recommendations = baseline_recommendations.sort_values(
    "baseline_score",
    ascending=False
)

baseline_rows = []
for k in top_ks:
    top_k = baseline_recommendations.head(k)
    hits = int(top_k["is_hidden_like"].sum())
    baseline_rows.append({
        "k": k,
        "baseline_hits": hits,
        "heldout_likes": len(heldout_ids),
        "baseline_hit_rate": hits / len(heldout_ids),
        "baseline_precision_at_k": hits / k,
    })

baseline_hit_rate_results = pd.DataFrame(baseline_rows)
hit_rate_results.merge(baseline_hit_rate_results, on=["k", "heldout_likes"])

,k,hits,heldout_likes,hit_rate,precision_at_k,baseline_hits,baseline_hit_rate,baseline_precision_at_k
0,5,2,31,0.064516,0.40,0,0.000000,0.00
1,10,2,31,0.064516,0.20,0,0.000000,0.00
2,20,5,31,0.161290,0.25,1,0.032258,0.05
3,50,6,31,0.193548,0.12,4,0.129032,0.08
4,100,9,31,0.290323,0.09,11,0.354839,0.11


Compare models

In [21]:
from anime_evaluation import HitRateEvaluator
 
n_runs = 50
compare_top_ks = (5, 10)
uncertainty_weight = 4.5

evaluator = HitRateEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    heldout_fraction=0.25,
)

compare_results, compare_summary = evaluator.compare_models(
    uncertainty_weight=uncertainty_weight,
    n_runs=n_runs,
    top_ks=compare_top_ks,
    include_lasso=True,
    include_elastic_net=True,
    random_state=42,
)

display(compare_summary)

ridge_alpha_counts = (
    compare_results.dropna(subset=["ridge_alpha"])["ridge_alpha"]
    .value_counts()
    .sort_index()
)
display(ridge_alpha_counts)

lasso_alpha_counts = (
    compare_results.dropna(subset=["lasso_alpha"])["lasso_alpha"]
    .value_counts()
    .sort_index()
)
display(lasso_alpha_counts)

elastic_param_counts = (
    compare_results.dropna(subset=["elastic_alpha", "elastic_l1_ratio"])
    .groupby(["elastic_alpha", "elastic_l1_ratio"])
    .size()
    .sort_index()
)
elastic_param_counts

,model,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits
0,bayesian_ridge,5,0.420,0.194831,0.067742,0.031424,2.10
8,ridge_cv,5,0.332,0.169561,0.053548,0.027349,1.66
2,elastic_net_cv,5,0.280,0.151186,0.045161,0.024385,1.40
6,lasso_cv,5,0.240,0.156492,0.038710,0.025241,1.20
4,global_mean,5,0.096,0.129300,0.015484,0.020855,0.48
1,bayesian_ridge,10,0.358,0.126314,0.115484,0.040746,3.58
9,ridge_cv,10,0.288,0.111831,0.092903,0.036074,2.88
3,elastic_net_cv,10,0.212,0.100285,0.068387,0.032350,2.12
7,lasso_cv,10,0.204,0.100934,0.065806,0.032559,2.04
5,global_mean,10,0.064,0.063116,0.020645,0.020360,0.64


ridge_alpha
351.119173      2
497.702356      2
559.081018      6
628.029144      6
705.480231     20
792.482898     16
890.215085     26
1000.000000    22
Name: count, dtype: int64

lasso_alpha
0.091030     4
0.115140    18
0.145635    78
Name: count, dtype: int64

elastic_alpha  elastic_l1_ratio
0.091030       0.9                  2
0.115140       0.9                  6
0.145635       0.7                  2
               0.9                 72
0.184207       0.9                 18
dtype: int64

## Results

Bayesian Ridge with uncertainty adjustment performed best across both cutoffs in the 50-run model comparison for user `chekkit`.

| Model | P@5 | P@10 | Avg hits@5 | Avg hits@10 |
| --- | ---: | ---: | ---: | ---: |
| bayesian_ridge | 0.420 | 0.358 | 2.10 | 3.58 |
| ridge_cv | 0.332 | 0.288 | 1.66 | 2.88 |
| elastic_net_cv | 0.280 | 0.212 | 1.40 | 2.12 |
| lasso_cv | 0.240 | 0.204 | 1.20 | 2.04 |
| global_mean | 0.096 | 0.064 | 0.48 | 0.64 |

The updated summary is saved in `metrics/model_comparison_20260615_153918.csv`.

Conclusion: keep **Bayesian Ridge** as the recommendation reranker. It beats RidgeCV, LassoCV, ElasticNetCV, and the global mean baseline at both Precision@5 and Precision@10.